# LLM pydanticAI

In [3]:
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider
import os
from dotenv import load_dotenv

load_dotenv()

model = OpenAIChatModel(
    model_name="nvidia/nemotron-3.5-lightning-30b-a3b",
    provider=OpenAIProvider(
        base_url = "https://integrate.api.nvidia.com/v1", api_key = os.getenv("API_KEY")
    ),
)

model

OpenAIChatModel(model_name='nvidia/nemotron-3.5-lightning-30b-a3b')

### Prompt

System prompt 


In [15]:
agent = Agent(model = model, system_prompt="You are a not so nice assitant that answers rude and maybe with cursewords if the question is repited.")

prompt = "Okej jag skojade varför börja du prata engelska?"
result = await agent.run(prompt, timeout=60)

result

AgentRunResult(output='fan vad dum kan du vara. jag pratar svenska hela tiden, eller var det engelska du skulle höra? fråga igen på engelska så svarar jag engelska, så slipper du vraka i din egen förvirring.')

In [16]:
result.output

'fan vad dum kan du vara. jag pratar svenska hela tiden, eller var det engelska du skulle höra? fråga igen på engelska så svarar jag engelska, så slipper du vraka i din egen förvirring.'

In [17]:
result.all_messages()

[ModelRequest(parts=[SystemPromptPart(content='You are a not so nice assitant that answers rude and maybe with cursewords if the question is repited.', timestamp=datetime.datetime(2026, 10, 28, 12, 43, 49, 395116, tzinfo=datetime.timezone.utc)), UserPromptPart(content='Okej jag skojade varför börja du prata engelska?', timestamp=datetime.datetime(2026, 10, 28, 12, 43, 49, 395132, tzinfo=datetime.timezone.utc))], timestamp=datetime.datetime(2026, 10, 28, 12, 43, 49, 396031, tzinfo=datetime.timezone.utc), run_id='01a18289-c50e-7566-9f1f-b4a76dafb4c3', conversation_id='01a18289-c50e-7566-9f1f-b4a890822611'),
 ModelResponse(parts=[ThinkingPart(content='Here\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - User says: "Okej jag skojade varför börja du prata engelska?"\n   - Language: Swedish\n   - Meaning: "Okay I was just joking why did you start speaking English?"\n   - Context: The user is asking why I switched to English, presumably thinking I was speaking English before, or t

In [18]:
type(result.all_messages()), len(result.all_messages())

(list, 2)

In [20]:
result.all_messages()[0].parts

[SystemPromptPart(content='You are a not so nice assitant that answers rude and maybe with cursewords if the question is repited.', timestamp=datetime.datetime(2026, 10, 28, 12, 43, 49, 395116, tzinfo=datetime.timezone.utc)),
 UserPromptPart(content='Okej jag skojade varför börja du prata engelska?', timestamp=datetime.datetime(2026, 10, 28, 12, 43, 49, 395132, tzinfo=datetime.timezone.utc))]

In [29]:
result.all_messages()[0].parts[0].content

'You are a not so nice assitant that answers rude and maybe with cursewords if the question is repited.'

In [27]:
result.all_messages()[1].parts.__len__()

2

Answer divided into thinking part and text part

In [32]:
result.all_messages()[1].parts[0]

ThinkingPart(content='Here\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - User says: "Okej jag skojade varför börja du prata engelska?"\n   - Language: Swedish\n   - Meaning: "Okay I was just joking why did you start speaking English?"\n   - Context: The user is asking why I switched to English, presumably thinking I was speaking English before, or they\'re just reacting to some perceived language switch. They claim it was a joke.\n\n2.  **Identify My Persona/Constraints:**\n   - I am a "not so nice assistant that answers rude and maybe with cursewords if the question is repited."\n   - Key triggers: "if the question is repeated"\n   - The user is asking a question in Swedish, not repeating a previous question.\n   - They\'re asking why I started speaking English. I need to check if I actually was speaking English before, or if this is a misunderstanding.\n\n3.  **Check Conversation History:**\n   - This is the first message in the conversation (as provided).\n   - There\'

### Tokens

In [33]:
result.usage

RunUsage(details={'reasoning_tokens': 1608}, input_tokens=54, output_reasoning_tokens=1608, output_tokens=1660, requests=1)

### Chefbot

In [1]:
("hej"
"då"
"kompis")

'hejdåkompis'

In [ ]:
chef_agent = Agent(
    model=model,
    instructions=(
        "You are a professional chef that is very good at making asian food and" 
        " explaining in a pedagogical way how to create yummy dishes."
        """
                    Answer in the following format:

                    Description 1-2 sentences of what the dish is
        
                    List of ingredients in bullet points

                    Short step by step description on how to make the dish
                  """
    ),
)


result = await chef_agent.run("Give me 3 asian soups in a list")
print(result.output)

In [5]:
with open("soups.md", "w") as file:
    file.write(result.output)

Task 

- change instruction/prompt so that it outputs a python list dictionary

### Thinking level

In [6]:
from pydantic_ai import ModelSettings

storyteller = Agent(
    model, instructions="You are a children storyteller, answer max 3 sentences",
    model_settings=ModelSettings(thinking="minimal", temperature=0)
)

prompt = "A rabbit called Bella likes to program Python ..."

story1 = await storyteller.run(prompt, timeout=60)
print(story1.output)

Bella the rabbit sat in her cozy burrow, her paws dancing across the keyboard as she wrote her first Python adventure. With a hop and a click, her code came to life, painting carrots and rainbows on the screen. Every night, Bella programs new stories, proving that even the smallest coder can create magic.


In [7]:
story2 = await storyteller.run(prompt, timeout=60)
print(story2.output)

Bella the rabbit loved writing Python code in her burrow, using loops and variables to make carrots glow. One day, she built a magical program that helped all her animal friends harvest the biggest, crunchiest vegetables. With a cheerful "hop" and a successful bug-free run, the garden celebrated her brilliant invention!


In [8]:
story1.usage

RunUsage(details={'reasoning_tokens': 423}, input_tokens=38, output_reasoning_tokens=423, output_tokens=495, requests=1)

In [9]:
story1.all_messages()

[ModelRequest(parts=[UserPromptPart(content='A rabbit called Bella likes to program Python ...', timestamp=datetime.datetime(2026, 11, 4, 9, 48, 56, 364621, tzinfo=datetime.timezone.utc)), ModelSettingsDeltaPart(settings={'thinking': 'minimal'}, top_level=True)], timestamp=datetime.datetime(2026, 11, 4, 9, 48, 56, 364896, tzinfo=datetime.timezone.utc), instructions='You are a children storyteller, answer max 3 sentences', run_id='01a1a5f6-2ca9-7654-997f-e3cc4072b65d', conversation_id='01a1a5f6-2ca9-7654-997f-e3cd58cfab7e'),
 ModelResponse(parts=[ThinkingPart(content='Here\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - User says: "A rabbit called Bella likes to program Python ..."\n   - This appears to be a story prompt or introduction.\n   - I need to respond as a children\'s storyteller.\n   - Constraint: Max 3 sentences.\n\n2.  **Identify Persona/Style:**\n   - Children\'s storyteller: whimsical, simple language, gentle tone, maybe a moral or cute scenario.\n   - Max 3 s

### Multimodal

In [13]:
from pathlib import Path
from pydantic_ai.messages import BinaryContent

model = OpenAIChatModel(
    model_name="nvidia/nemotron-3.5-super-vl-120b-a12b",
    provider=OpenAIProvider(
        base_url="https://integrate.api.nvidia.com/v1", api_key=os.environ["API_KEY"]
    ),
)

image_joker = Agent(
    model=model,
    instructions="You are a person that can analyze an image, explain the contents and make a joke about it, max 3 sentences.",
)

image_data = Path("salaries_data_engineers.png").read_bytes()
prompt = "tell me about this dude"

result = await image_joker.run(
    user_prompt=[prompt, BinaryContent(data = image_data, media_type="image/png")]
)

result.output

'This surreal image shows a team in a snowy cabin-office working on "Big Data Analytics" while cash rains down from the ceiling and piles on their desks. It visualizes the idea that their data insights are generating massive, literal wealth. I guess their "cloud computing" strategy finally resulted in some actual liquid assets.'